# 01 · On-Chain Hashrate & Difficulty

**Purpose:** Establish the on-chain supply-side foundation of the mining model.
Hashrate and difficulty are the primary inputs to:
- Hashprice calculation (notebook 04)
- Mining company valuation (notebook 05)
- Factor model (notebook 06)

**Data sources:**
- Primary: [CoinMetrics Community API](https://coinmetrics.io/community-network-data/) — free, no auth
- Optional: `bigquery-public-data.crypto_bitcoin.blocks` — requires GCP billing project

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.coinmetrics import fetch_hashrate_difficulty, fetch_btc_price
from src.data import bigquery_client
from src.models.hashprice import estimate_hashrate_from_difficulty, difficulty_ribbon
from src.utils.plotting import (
    plot_hashrate_with_difficulty, set_matplotlib_style, PLOTLY_TEMPLATE, BTC_ORANGE
)
from config import DEFAULT_LOOKBACK_DAYS

pd.options.display.float_format = '{:,.2f}'.format
print('Setup complete.')

## 1. Fetch Hashrate & Difficulty Data

In [ ]:
# Fetch 2 years of daily on-chain metrics from CoinMetrics
onchain = fetch_hashrate_difficulty(days=DEFAULT_LOOKBACK_DAYS)
print(f'Fetched {len(onchain)} days of data ({onchain.index[0].date()} → {onchain.index[-1].date()})')
onchain.tail()

In [ ]:
# Rename for convenience
hashrate = onchain['hashrate']         # EH/s
difficulty = onchain['difficulty']     # network difficulty
block_count = onchain['blocks']        # blocks mined per day
fees_btc = onchain['fees_btc']         # total fees in BTC per day

# Derived: average fees per block
fees_per_block = (fees_btc / block_count).rename('fees_per_block_btc')

print(f'Latest hashrate:    {hashrate.iloc[-1]:.1f} EH/s')
print(f'Latest difficulty:  {difficulty.iloc[-1]:,.0f}')
print(f'Avg blocks/day:     {block_count.mean():.1f}')
print(f'Avg fees/block:     {fees_per_block.mean():.4f} BTC')

## 2. Hashrate & Difficulty Chart

In [ ]:
fig = plot_hashrate_with_difficulty(hashrate, difficulty)
fig.show()

## 3. Hashrate Moving Averages

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hashrate.index, y=hashrate.values,
    name='Daily Hashrate', mode='lines',
    line=dict(color='rgba(247,147,26,0.4)', width=1),
))

for window, color in [(14, '#F39C12'), (30, BTC_ORANGE), (90, '#E74C3C'), (200, '#9B59B6')]:
    ma = hashrate.rolling(window).mean()
    fig.add_trace(go.Scatter(
        x=ma.index, y=ma.values,
        name=f'{window}d MA', mode='lines',
        line=dict(color=color, width=2),
    ))

fig.update_layout(
    title='Bitcoin Network Hashrate with Moving Averages',
    yaxis_title='Hashrate (EH/s)', xaxis_title='Date',
    template=PLOTLY_TEMPLATE, height=450, hovermode='x unified',
)
fig.show()

## 4. Difficulty Adjustment Analysis

Bitcoin adjusts difficulty every 2016 blocks (~2 weeks) to target a 10-minute block time.
Difficulty increases when hashrate rises (miners are faster than the target) and decreases when hashrate falls.

In [ ]:
# Difficulty % change period-over-period (each adjustment epoch)
# Approximate: sample difficulty once per 14 days
diff_biweekly = difficulty.resample('14D').last()
diff_pct_change = diff_biweekly.pct_change() * 100

fig = go.Figure(go.Bar(
    x=diff_pct_change.index,
    y=diff_pct_change.values,
    marker_color=[BTC_ORANGE if v >= 0 else '#E74C3C' for v in diff_pct_change.fillna(0)],
    name='Difficulty Change %',
))
fig.add_hline(y=0, line_color='white', line_width=0.5)
fig.update_layout(
    title='Bitcoin Difficulty Adjustment (per ~2-week epoch)',
    yaxis_title='% Change per Epoch', xaxis_title='Date',
    template=PLOTLY_TEMPLATE, height=400,
)
fig.show()

print(f'\nLargest single increase: {diff_pct_change.max():.1f}%')
print(f'Largest single decrease: {diff_pct_change.min():.1f}%')
print(f'Average adjustment:      {diff_pct_change.mean():.2f}%')

## 5. Difficulty Ribbon

The **difficulty ribbon** is a set of SMAs of the hashrate. When faster SMAs cross below slower ones
(ribbon compression), it historically signals miner capitulation — a potential accumulation zone.

In [ ]:
ribbon = difficulty_ribbon(hashrate)

fig = go.Figure()
colors_ribbon = ['#F7931A','#F39C12','#E67E22','#E74C3C','#9B59B6','#3498DB','#27AE60','#1ABC9C']

fig.add_trace(go.Scatter(
    x=hashrate.index, y=hashrate.values,
    name='Hashrate', mode='lines',
    line=dict(color='rgba(255,255,255,0.3)', width=1),
))

for (col, sma), color in zip(ribbon.items(), colors_ribbon):
    fig.add_trace(go.Scatter(
        x=sma.index, y=sma.values,
        name=col, mode='lines',
        line=dict(color=color, width=1.5),
    ))

fig.update_layout(
    title='Bitcoin Difficulty Ribbon (Hashrate SMAs)',
    yaxis_title='EH/s', xaxis_title='Date',
    template=PLOTLY_TEMPLATE, height=500, hovermode='x unified',
)
fig.show()

## 6. Blocks Per Day & Block Time Analysis

In [ ]:
avg_block_time_min = (24 * 60) / block_count  # minutes per block

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Blocks Per Day (target=144)', 'Avg Block Time (min, target=10)'])

fig.add_trace(go.Scatter(
    x=block_count.index, y=block_count.rolling(7).mean(),
    name='Blocks/day (7d MA)', line=dict(color=BTC_ORANGE, width=2),
), row=1, col=1)
fig.add_hline(y=144, line_dash='dash', line_color='gray', row=1, col=1)

fig.add_trace(go.Scatter(
    x=avg_block_time_min.index, y=avg_block_time_min.rolling(7).mean(),
    name='Block time (7d MA)', line=dict(color='#3498DB', width=2),
), row=2, col=1)
fig.add_hline(y=10, line_dash='dash', line_color='gray', row=2, col=1)

fig.update_layout(template=PLOTLY_TEMPLATE, height=500, showlegend=False)
fig.show()

## 7. Hashrate From Difficulty (Cross-check)

We can independently estimate hashrate from difficulty using:
```
H (EH/s) = difficulty × 2^32 / (avg_block_time_s × 10^18)
```

In [ ]:
avg_block_time_s = (24 * 3600) / block_count  # seconds per block
hashrate_from_diff = difficulty.combine(avg_block_time_s, estimate_hashrate_from_difficulty)

# Vectorized version
hashrate_derived = (difficulty * (2**32)) / (avg_block_time_s * 1e18)
hashrate_derived.name = 'hashrate_derived_ehs'

fig = go.Figure()
fig.add_trace(go.Scatter(x=hashrate.index, y=hashrate.values,
                          name='CoinMetrics', line=dict(color=BTC_ORANGE, width=2)))
fig.add_trace(go.Scatter(x=hashrate_derived.index, y=hashrate_derived.values,
                          name='Derived from Difficulty', line=dict(color='#3498DB', width=2, dash='dot')))
fig.update_layout(title='Hashrate: CoinMetrics vs. Derived from Difficulty',
                   yaxis_title='EH/s', template=PLOTLY_TEMPLATE, height=400)
fig.show()

## 8. Optional: BigQuery Analysis

If you have a GCP billing project, uncomment the cells below to query `bigquery-public-data.crypto_bitcoin.blocks` directly.

In [ ]:
# import os
# os.environ['GCP_PROJECT'] = 'your-billing-project-id'
# 
# if bigquery_client.is_available():
#     bq_hashrate = bigquery_client.fetch_block_hashrate(days=365)
#     print(bq_hashrate.tail())
# else:
#     print('BigQuery not configured. Set GCP_PROJECT env var.')
print('BigQuery section: uncomment and set GCP_PROJECT to enable.')

## Summary

| Metric | Value |
|--------|-------|
| Current Hashrate | See above |
| 90d Change | See chart |
| Difficulty ATH | See chart |
| Avg Fees/Block | See above |

**Key takeaways:**
- Bitcoin's hashrate has grown dramatically since the 2024 halving, reflecting increased miner capital deployment
- Difficulty adjustments track hashrate closely, maintaining the 10-min target block time
- The difficulty ribbon can signal miner capitulation cycles

**Next:** [02 · BTC Price & Perpetual Swaps →](./02_btc_price_perp_swaps.ipynb)